# Car Sales Data Preparation for Power BI

This notebook documents the reproducible data-cleaning steps used to prepare the car-sales dataset for dashboard analysis. The original dataset contains 157 vehicle records and 15 source columns.

The cleaning workflow preserves the original project logic: placeholder missing values are converted to `NaN`, numeric fields are coerced to numeric types, missing numeric values are filled with column medians, and a three-level price category is created.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


## Load the source dataset

Place `Car_sales.csv` inside the repository's `data/` directory before running the notebook.

In [ ]:
DATA_PATH = Path('data/Car_sales.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing {DATA_PATH}. See data/README.md for the expected input file.'
    )

car_data = pd.read_csv(DATA_PATH)
print('Dataset shape:', car_data.shape)
display(car_data.head())


## Clean invalid and missing numeric values

The original dataset uses values such as `.`, `NA`, `N/A`, and `na` as placeholders. These are converted to missing values before numeric conversion.

In [ ]:
car_data = car_data.replace(['.', 'NA', 'N/A', 'na'], np.nan)

numeric_cols = [
    '4-year resale value',
    'Price in thousands',
    'Engine size',
    'Horsepower',
    'Wheelbase',
    'Width',
    'Length',
    'Curb weight',
    'Fuel capacity',
    'Fuel efficiency',
]

for column in numeric_cols:
    car_data[column] = pd.to_numeric(car_data[column], errors='coerce')
    car_data[column] = car_data[column].fillna(car_data[column].median())

car_data[numeric_cols].isna().sum()


## Create price categories

Vehicles are grouped into three price bands using the same thresholds as the original analysis.

In [ ]:
car_data['Price Category'] = pd.cut(
    car_data['Price in thousands'],
    bins=[0, 20, 35, np.inf],
    labels=['Low', 'Medium', 'High'],
)

display(car_data.head())
car_data['Price Category'].value_counts(dropna=False)


## Export the cleaned dataset

The cleaned CSV can be loaded into Power BI as a reproducible input table.

In [ ]:
OUTPUT_PATH = Path('data/car_sales_cleaned.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
car_data.to_csv(OUTPUT_PATH, index=False)

print(f'Saved cleaned dataset to: {OUTPUT_PATH}')
print('Final shape:', car_data.shape)
